# 异常
`Starlette` 允许您安装自定义异常处理程序来处理在发生错误或处理异常时如何返回响应。


In [ ]:
from starlette.applications import Starlette
from starlette.exceptions import HTTPException
from starlette.requests import Request
from starlette.responses import HTMLResponse


HTML_404_PAGE = ...
HTML_500_PAGE = ...


async def not_found(request: Request, exc: HTTPException):
    return HTMLResponse(content=HTML_404_PAGE, status_code=exc.status_code)

async def server_error(request: Request, exc: HTTPException):
    return HTMLResponse(content=HTML_500_PAGE, status_code=exc.status_code)


exception_handlers = {
    404: not_found,
    500: server_error
}

app = Starlette(routes=routes, exception_handlers=exception_handlers)

如果debug启用并且发生错误，那么 Starlette 将使用回溯响应进行响应，而不是使用已安装的 500 处理程序。

In [1]:
app = Starlette(debug=True, routes=routes, exception_handlers=exception_handlers)

NameError: name 'Starlette' is not defined

除了为特定状态代码注册处理程序之外，您还可以为异常类注册处理程序。

你可能希望覆盖内置类的`HTTPException`处理方式。例如，要使用 JSON 样式的响应：

In [ ]:
async def http_exception(request: Request, exc: HTTPException):
    return JSONResponse({"detail": exc.detail}, status_code=exc.status_code)

exception_handlers = {
    HTTPException: http_exception
}

HTTPException还配备了参数。它headers允许将标头传播到响应类：

In [ ]:
async def http_exception(request: Request, exc: HTTPException):
    return JSONResponse(
        {"detail": exc.detail},
        status_code=exc.status_code,
        headers=exc.headers
    )

您可能还想覆盖WebSocketException处理方式：

In [ ]:
async def websocket_exception(websocket: WebSocket, exc: WebSocketException):
    await websocket.close(code=1008)

exception_handlers = {
    WebSocketException: websocket_exception
}

## 错误和已处理的异常

区分已处理的异常和错误非常重要。

已处理的异常不表示错误情况。它们被强制转换为适当的 HTTP 响应，然后通过标准中间件堆栈发送。默认情况下，HTTPException 类用于管理任何已处理的异常。

错误是应用程序中发生的任何其他异常。这些情况应该作为异常在整个中间件堆栈中冒泡。任何错误日志记录中间件都应确保它重新引发异常，一直到服务器。

实际上，使用处理的错误是 `exception_handler[500]` 或 `exception_handler[Exception]`。键 `500` 和 `Exception` 都可以使用。请参阅下文：

In [ ]:
async def handle_error(request: Request, exc: HTTPException):
    # Perform some logic
    return JSONResponse({"detail": exc.detail}, status_code=exc.status_code)

exception_handlers = {
    Exception: handle_error  # or "500: handle_error"
}

请务必注意，如果 `BackgroundTask` 引发异常，它将由 `handle_error` 函数处理，但此时，响应已发送。换句话说，`handle_error` 创建的响应将被丢弃。如果错误发生在发送响应之前，则它将使用响应对象 - 在上面的示例中，返回的 `JSONResponse`。

为了正确处理这种行为， Starlette应用程序的中间件堆栈配置如下：
- ServerErrorMiddleware- 当服务器发生错误时返回 500 响应。
- 安装的中间件
- ExceptionMiddleware- 处理已处理的异常并返回响应。
- 路由器
- 端点

## HTTPException

`HTTPException` 类提供了一个基类，您可以将其用于任何已处理的异常。`ExceptionMiddleware` 实现默认为任何 `HTTPException` 返回纯文本 HTTP 响应。

- `HTTPException(status_code, detail=None, headers=None)`

您只应在路由或端点内部引发 `HTTPException`。中间件类应直接返回适当的响应。
您可以在 `WebSocket` 终端节点上使用 `HTTPException`。如果它是在 `websocket.accept()` 之前引发的，则连接不会升级为 `WebSocket` 连接，并返回正确的 HTTP 响应。

In [ ]:
from starlette.applications import Starlette
from starlette.exceptions import HTTPException
from starlette.routing import WebSocketRoute
from starlette.websockets import WebSocket


async def websocket_endpoint(websocket: WebSocket):
    raise HTTPException(status_code=400, detail="Bad request")


app = Starlette(routes=[WebSocketRoute("/ws", websocket_endpoint)])

## WebSocketException
您可以使用该类`WebSocketException`在 `WebSocket` 端点内引发错误。

- `WebSocketException(code=1008, reason=None)`

您可以设置任何符合规范定义的有效代码。